# Lab 1: Primer

Comparisons, intervals, and fake-data checks

You already know how to fit a line. This lab asks you to write what the
slope is allowed to mean, to put an interval next to a point estimate,
and to test your usual $t$-procedure on data where the truth is known.
There is no submission. Bring the write-up back to the seminar.

<figure>
<a
href="https://colab.research.google.com/github/quinfer/fin510-colab-notebooks/blob/main/labs/lab01_primer.ipynb"><img
src="https://colab.research.google.com/assets/colab-badge.svg" /></a>
<figcaption>Open in Colab</figcaption>
</figure>

Core: about an hour. Optional logistic extension: another 20 minutes.
Stylised-fact plots and bias-variance cartoons belong to later weeks; do
not spend this session on them.

In [ ]:
# Colab only: uncomment if imports fail
# %pip -q install numpy pandas matplotlib scipy statsmodels
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy import stats

## Orientation

This cell checks that the kernel can do the operations the rest of the
lab uses. If an assertion fails, fix the environment before going on.

In [ ]:
print("Python is running")

x = 10
assert x == 10, "Variable assignment failed"

numbers = [1, 2, 3, 4, 5]
assert numbers[0] == 1
assert numbers[-1] == 5
assert numbers[1:3] == [2, 3]

stock = {"ticker": "AAPL", "price": 150.0, "volume": 1000000}
assert stock["ticker"] == "AAPL"
assert stock["price"] == 150.0

def add_returns(r1, r2):
    return r1 + r2

assert add_returns(0.05, 0.03) == 0.08
print("Orientation checks passed")

`assert` stops the notebook at the first false claim. That is the point:
fail loudly here rather than silently in a later cell.

## Task 1: Slope as a comparison

The chapter’s ESG regression is a teaching equation, not a finding from
a ratings file. Here the data-generating process is known, so you can
see what OLS reports and what it cannot.

Firms have an unobserved quality factor. ESG scores and returns both
load on that factor. Market cap is an extra covariate. You fit returns
on ESG and market cap: the same specification as the slide, with a
different numerical slope because this is a new draw.

In [ ]:
rng = np.random.default_rng(42)
n = 400
quality = rng.normal(0, 1, n)
esg = quality + rng.normal(0, 1, n)
mcap = rng.normal(0, 1, n)
annual_return = 8.2 + 1.5 * quality + 0.02 * mcap + rng.normal(0, 5, n)

df = pd.DataFrame({"ret": annual_return, "esg": esg, "mcap": mcap})
X = sm.add_constant(df[["esg", "mcap"]])
ols = sm.OLS(df["ret"], X).fit()
hc = sm.OLS(df["ret"], X).fit(cov_type="HC1")
print(ols.summary().tables[1])
print("ESG slope (default SE): {:.3f} ({:.3f})".format(ols.params["esg"], ols.bse["esg"]))
print("ESG slope (HC1 SE):     {:.3f} ({:.3f})".format(hc.params["esg"], hc.bse["esg"]))

The coefficient on ESG is a between-firm comparison: firms with higher
ESG scores had, on average, higher returns than otherwise similar firms
in this sample, holding the included market-cap term fixed (Gelman,
Hill, and Vehtari 2020). It is not the 1.5 loading on *quality*. Quality
is not in the file. Copying another firm’s ESG score, without copying
quality, would not deliver that 1.5.

Default OLS standard errors assume constant residual variance. HC1
relaxes that. In a cross-section, HC is the usual patch; HAC is for
calendar-time residuals (Week 3). Compare the two ESG standard errors
above. If they barely move, heteroscedasticity is not the live problem
in this draw. If they do move, the default $t$ was the thing that
changed, not the slope.

**Write two sentences in your notes.** First: the slope as a comparison.
Second: what identification you would need before calling it an effect
(a within-firm intervention, or an instrument that shifts ESG without
shifting quality).

**Discussion questions:**

1.  Why is the fitted ESG slope not equal to 1.5 even though 1.5 is in
    the data-generating process?
2.  A colleague says “the HC errors did not change the coefficient, so
    they were unnecessary.” What did they mix up?
3.  Would HAC standard errors be the right patch for this file?

## Task 2: Estimate and interval

The lecture’s rule is to report the estimate and an interval, then ask
whether the number is large enough to trade. A percentile interval from
resampling the same file is one way to get that interval without
assuming a Gaussian for the raw observations. It is not a new sampling
theory for the exam.

In [ ]:
rng = np.random.default_rng(3)
x = rng.lognormal(mean=0.0, sigma=0.5, size=300)
B = 2000
boot_means = np.array([np.mean(rng.choice(x, size=len(x), replace=True)) for _ in range(B)])
ci_low, ci_high = np.percentile(boot_means, [2.5, 97.5])
xbar = np.mean(x)

print(f"Point estimate: {xbar:.4f}")
print(f"95% percentile interval: [{ci_low:.4f}, {ci_high:.4f}]")
assert ci_low < xbar < ci_high

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(boot_means, bins=40, color="steelblue", edgecolor="white", alpha=0.85)
ax.axvline(ci_low, color="coral", linestyle="--", lw=2)
ax.axvline(ci_high, color="coral", linestyle="--", lw=2)
ax.axvline(xbar, color="black", lw=2)
ax.set_xlabel("Resampled mean")
ax.set_ylabel("Count")
fig.tight_layout()
plt.show()

A frequentist 95% interval is a claim about the procedure: in repeated
samples, 95% of such intervals contain the mean. It is not a 95%
probability that this particular interval covers the mean. The lecture’s
Bayesian credible interval on daily SPY $R^2$ (posterior median 1.66%,
95% interval 0.08% to 6.24%) is a different object: a posterior for a
parameter, given a prior and a likelihood. Do not treat the two as
interchangeable.

**Discussion questions:**

1.  The lecture strategy had mean 0.001% and SE 0.0003%. How would you
    use an interval, rather than $t \approx 3.3$, to decide whether it
    is dust after costs?
2.  Why does a short window make a normal-based interval for a mean too
    narrow even when the CLT is the reason $t$ tests still work on
    means?

## Task 3: Later window, not a shuffle

Financial returns are ordered in time. Leave-one-out and shuffled
$K$-fold treat rows as exchangeable. Shuffling days leaks the future
into the past (Efron and Hastie 2016). This week’s rule is shorter than
Chapter 10: the test set is strictly after the training set.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 2.2))
blocks = [(0, 40, "Train", "steelblue"), (40, 55, "Test", "coral")]
for start, end, label, color in blocks:
    ax.barh(0, end - start, left=start, height=0.5, color=color, edgecolor="black")
    ax.text((start + end) / 2, 0, label, ha="center", va="center", color="white")
ax.set_yticks([])
ax.set_xlabel("Time index")
ax.set_xlim(-2, 58)
ax.set_title("One later window (schematic)")
fig.tight_layout()
plt.show()

Walk-forward, CSCV, and probability of backtest overfitting are Week 10.
Do not import them into Exercise 1.

**Discussion question:** A classmate proposes five-fold CV after
randomly shuffling daily SPY returns. What leaks, and what would you do
instead this week?

## Task 4: Fake-data alpha

The chapter’s test: if you cannot detect a planted alpha with your usual
$t$-test, you should not claim one in the wild. Plant two annual alphas
in ten years of daily Gaussian draws, run a one-sample $t$-test against
zero, and repeat.

Daily mean is annual alpha divided by 252. Daily standard deviation is
1% (about 16% a year). Ten years is 2,520 days.

In [ ]:
def detection_rate(annual_alpha, n_days=2520, daily_vol=0.01, n_sims=400, seed=7):
    rng = np.random.default_rng(seed)
    mu = annual_alpha / 252
    reject = 0
    for _ in range(n_sims):
        r = rng.normal(mu, daily_vol, n_days)
        p = stats.ttest_1samp(r, popmean=0.0).pvalue
        if p < 0.05:
            reject += 1
    return reject / n_sims

for a in (0.00, 0.01, 0.20):
    rate = detection_rate(a)
    print(f"Annual alpha {a:.0%}: rejection rate at 5% = {rate:.2f}")

A 1% annual alpha in this design is dust relative to daily noise: the
rejection rate stays near the 5% size of the test. A 20% annual alpha is
a different object; detection rises. That is why a $t$ of 3.3 on 0.001%
excess return can still be economically empty, and why fake-data is a
check on the *procedure*, not a replacement for market data.

**Discussion questions:**

1.  Why is the rejection rate under a true zero near 0.05, and why is
    that a calibration check rather than a finding about markets?
2.  The lecture’s 0.001% example had $t \approx 3.3$. What must be
    different from this 1% annual design: the mean, the volatility, or
    the sample size?

## Optional: logistic divide-by-four

A debt-to-income coefficient of 0.4 on the logit scale is at most a 10
percentage-point change in probability, and that bound is tightest at
probability 0.5 (Gelman, Hill, and Vehtari 2020). Away from 0.5 the
slope is smaller. For a rare event, evaluate the predicted probability
at two values of $X$ instead of quoting $\beta/4$ as if it were exact.

In [ ]:
beta = 0.4
print(f"Divide-by-four bound: {beta / 4:.2f} (10 pp)")

def logit_p(x, b0, b1=0.4):
    z = b0 + b1 * x
    return 1 / (1 + np.exp(-z))

# b0 chosen so that P = 0.5 when x = 0, and a rare-event intercept
for name, b0 in [("at p=0.5", 0.0), ("rare event", -4.0)]:
    p0 = logit_p(0.0, b0)
    p1 = logit_p(1.0, b0)
    print(f"{name}: P(x=0)={p0:.3f}, P(x=1)={p1:.3f}, change={p1 - p0:.3f}")

Compare the two changes with 0.10. Only the probability-0.5 line
approaches the bound.

## Directed learning

-   Read Gelman, Hill, and Vehtari (2020) on assumptions ranked by
    importance, coefficients as comparisons, and fake-data simulation
    (the chapters your handbook lists for Week 1).
-   Keep the two-sentence OLS write-up. That is the artefact to bring to
    class.
-   Optional: change the planted alpha and the number of years in Task 4
    and record how detection moves.
-   [Week 1 practice items](../practice/week01_foundations.qmd): twelve
    application questions, unlimited attempts. Not the Exercise 1 paper.

Week 2 takes the same discipline to a file: where the numbers come from,
who dropped out, and whether a plot of SPY shows the stylised facts.

## References

Efron, Bradley, and Trevor Hastie. 2016. *Computer Age Statistical
Inference: Algorithms, Evidence, and Data Science*. Cambridge University
Press. <https://hastie.su.stanford.edu/CASI/>.

Gelman, Andrew, Jennifer Hill, and Aki Vehtari. 2020. *Regression and
Other Stories*. Cambridge, UK: Cambridge University Press.
<https://avehtari.github.io/ROS-Examples/>.